# Notebook 1 — Layer Calibration

Identifies two layer boundaries before any concept extraction begins:

| Symbol | Method | Meaning |
|---|---|---|
| `L_det` | Token Erasure (footprints probes) | Layer where multi-token SNOMED concepts have been merged into a single representation |
| `L_pred` | Logit Lens | Layer where final-layer next-token predictions stabilise |

**Inputs**
- Model already loaded via notebook 0 (`src/config.py` → `MODEL_NAME`)
- `data/embeddings-concept-openai/concepts.csv` — breast concept preferred terms
- Pretrained probes from `sfeucht/footprints` on HF Hub (downloaded automatically for `llama-3-8b`)

**Outputs**
- `data/stage1/layer_boundaries.json` → `{"L_det": int, "L_pred": int, "model": str, "failure_rate_pct": float}`
- Layer-by-layer plots saved to `data/stage1/plots/`

**Blocked by:** notebook 0 (model must load successfully)

In [ ]:
# parameters
DATA_DIR     = "../../data/stage1"
CONCEPTS_CSV = "../../data/embeddings-concept-openai/concepts.csv"
N_SAMPLE     = 75    # number of concepts to run probes on (50–100 per plan)
SEED         = 42

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformer_lens import HookedTransformer

sys.path.insert(0, "../..")
sys.path.insert(0, "../../Reference_Papers/footprints/scripts")

from src.config import MODEL_NAME, TORCH_DTYPE

os.makedirs(os.path.join(DATA_DIR, "plots"), exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"dtype: {TORCH_DTYPE}")

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME, dtype=TORCH_DTYPE)
model.eval()

n_layers = model.cfg.n_layers
d_model  = model.cfg.d_model
print(f"n_layers: {n_layers}, d_model: {d_model}")

In [ ]:
rng = np.random.default_rng(SEED)
concepts_df = pd.read_csv(CONCEPTS_CSV)
sample = concepts_df.sample(n=min(N_SAMPLE, len(concepts_df)), random_state=SEED)
concept_names = sample["preferred_term"].tolist()

print(f"Sampled {len(concept_names)} concepts")
print("Examples:", concept_names[:3])

## Part 1 — Token Erasure → `L_det`

The footprints repo (`Reference_Papers/footprints/`) provides pretrained linear probes for Llama-3-8B on HF Hub (`sfeucht/footprints`). Each probe at layer `L` predicts token probabilities from the residual stream at that layer.

**Simplified erasure score used here:**
For a concept with tokens `[t_0, ..., t_{n-1}]`, we treat the concept as the target span. At each layer `L`, we take the hidden state at the last token position and apply the probe. The score is the probe's assigned probability to the first token of the concept — a proxy for how much the final position "contains" the whole concept (i.e., how much detokenization has occurred).

High score = concept has been merged into a unified representation → this peaks at `L_det`.

**Alternative:** The full `psi()` pipeline from `readout.py` requires `nnsight.LanguageModel`. If you want to run the exact paper algorithm, replace the probe application below with `get_doc_info()` + `psi()` from that file.

In [ ]:
# Import pretrained probe loader from footprints
from readout import get_probe

# Probe model name as used on HF Hub
PROBE_MODEL = "llama-3-8b"   # sfeucht/footprints naming convention

# Load probes for all layers (target index 'in1' = predict the next-token from current pos)
# This downloads ~few MB per layer; cached by huggingface_hub after first run
print("Loading probes (downloads on first run)...")
probes = {}
for L in range(n_layers):
    try:
        probes[L] = get_probe(L, "in1", PROBE_MODEL)
        probes[L].eval()
    except Exception as e:
        print(f"  Layer {L}: probe unavailable ({e})")

print(f"Loaded {len(probes)} / {n_layers} probes")

In [ ]:
layer_erasure_scores = {L: [] for L in probes}
failed_concepts = []
concept_caches  = {}   # keyed by concept name; reused in Part 2
concept_finals  = {}   # final-layer argmax per concept; reused in Part 2

for concept in concept_names:
    tokens = model.to_tokens(concept)             # (1, n_tokens)
    token_ids = tokens[0].tolist()
    n_tokens = len(token_ids)

    if n_tokens < 2:
        # Single-token concept — not informative for erasure
        failed_concepts.append(concept)
        continue

    first_token_id = token_ids[0]

    with torch.no_grad():
        final_logits, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name.endswith("hook_resid_post")
        )

    # Store for reuse in Part 2 — avoids a second forward pass per concept
    concept_caches[concept] = cache
    concept_finals[concept] = final_logits[0, -1, :].argmax().item()

    for L, probe in probes.items():
        h_last       = cache[f"blocks.{L}.hook_resid_post"][0, -1, :]  # (d_model,)
        probe_logits = probe(h_last.unsqueeze(0))                       # (1, vocab_size)
        probs        = torch.softmax(probe_logits.float(), dim=-1)
        score        = probs[0, first_token_id].item()
        layer_erasure_scores[L].append(score)

failure_rate = len(failed_concepts) / len(concept_names) * 100
print(f"Single-token concepts (skipped): {len(failed_concepts)} ({failure_rate:.1f}%)")
print("Note: paper reports ~23% failure rate for general English; higher expected for SNOMED terminology.")

In [ ]:
layers = sorted(layer_erasure_scores.keys())
mean_scores = [np.mean(layer_erasure_scores[L]) if layer_erasure_scores[L] else 0.0 for L in layers]

L_det = layers[int(np.argmax(mean_scores))]
print(f"L_det = {L_det}  (peak mean erasure score: {max(mean_scores):.4f})")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(layers, mean_scores, marker="o", markersize=4)
ax.axvline(L_det, color="red", linestyle="--", label=f"L_det = {L_det}")
ax.set_xlabel("Layer")
ax.set_ylabel("Mean erasure score")
ax.set_title("Token Erasure by Layer — SNOMED Breast Concepts")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, "plots", "token_erasure_by_layer.png"), dpi=150)
plt.show()
print(f"Saved: {DATA_DIR}/plots/token_erasure_by_layer.png")

## Part 2 — Logit Lens → `L_pred`

At each layer `L`, apply the model's final layer norm + unembedding to the residual stream and take argmax. `L_pred` is the earliest layer where predicted tokens match the final-layer predictions at a high rate (threshold: ≥ 80% of concepts agree).

**Why Logit Lens here (not Tuned Lens):**
Tuned Lens requires a HuggingFace `PreTrainedModel` (not `HookedTransformer`) and separate pretrained affine translator weights. Check `available_lens_artifacts()` from `tuned_lens.load_artifacts` if you want the tuned variant — if a checkpoint exists for Llama-3-8B, load it via `TunedLens.from_model_and_pretrained(hf_model)` for a more accurate `L_pred` estimate.

The Logit Lens provides a conservative (typically lower) estimate of `L_pred`; Tuned Lens typically finds a lower layer. Both bracket a reasonable range.

In [ ]:
MATCH_THRESHOLD = 0.80   # fraction of concepts that must agree with final layer

# Reuse caches from Part 1 — no additional forward passes needed.
# For concepts that were skipped (single-token), they are absent from concept_caches.
cached_concepts = list(concept_caches.keys())

# Per-concept, per-layer predictions: shape (n_cached_concepts, n_layers)
layer_preds = np.empty((len(cached_concepts), n_layers), dtype=np.int64)

for i, concept in enumerate(cached_concepts):
    cache = concept_caches[concept]
    with torch.no_grad():
        for L in range(n_layers):
            h_L        = cache[f"blocks.{L}.hook_resid_post"][0, -1, :]
            h_L_normed = model.ln_final(h_L.unsqueeze(0).unsqueeze(0))[0, 0, :]
            logits_L   = model.unembed(h_L_normed.unsqueeze(0).unsqueeze(0))[0, 0, :]
            layer_preds[i, L] = logits_L.argmax().item()

finals = np.array([concept_finals[c] for c in cached_concepts])   # (n_cached_concepts,)

# Match rate at each layer: fraction of concepts where layer-L pred == final-layer pred
layer_match_rates = (layer_preds == finals[:, None]).mean(axis=0).tolist()

exceeds = [L for L, r in enumerate(layer_match_rates) if r >= MATCH_THRESHOLD]
L_pred  = exceeds[0] if exceeds else n_layers - 1
print(f"L_pred = {L_pred}  (first layer with ≥{MATCH_THRESHOLD*100:.0f}% match rate: {layer_match_rates[L_pred]:.3f})")
print(f"(Evaluated on {len(cached_concepts)} multi-token concepts)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(n_layers), layer_match_rates, marker="o", markersize=4)
ax.axvline(L_pred, color="blue", linestyle="--", label=f"L_pred = {L_pred}")
ax.axhline(MATCH_THRESHOLD, color="grey", linestyle=":", label=f"Threshold = {MATCH_THRESHOLD}")
ax.set_xlabel("Layer")
ax.set_ylabel("Match rate with final-layer prediction")
ax.set_title("Logit Lens Match Rate by Layer — SNOMED Breast Concepts")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, "plots", "logit_lens_by_layer.png"), dpi=150)
plt.show()
print(f"Saved: {DATA_DIR}/plots/logit_lens_by_layer.png")

## Part 3 — Save layer boundaries

Write calibration results to `data/stage1/layer_boundaries.json`. **After this cell, manually update `src/config.py` with the `L_DET` and `L_PRED` values** so that notebook 2 picks them up.

In [ ]:
results = {
    "model": MODEL_NAME,
    "L_det": L_det,
    "L_pred": L_pred,
    "failure_rate_pct": round(failure_rate, 2),
    "n_concepts_sampled": len(concept_names),
    "match_threshold": MATCH_THRESHOLD,
}

out_path = os.path.join(DATA_DIR, "layer_boundaries.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print(f"\nSaved: {out_path}")
print(f"\nNext step: update src/config.py → L_DET = {L_det}, L_PRED = {L_pred}")